실습 5. asfreq와 resample 결측 비교
- 같은 정규화라도 결측 수가 왜 다른지 비교

목표
- 같은 10초 정규화라도 asfreq와 resample의 결측이 왜 다른지 비교

단계
- 같은 데이터에 격자 올리기와 구간 묶기를 각각 적용
- 두 결과를 나란히 두고 빈 칸 수를 비교
- 1분 다운샘플링으로 결측이 크게 줄어듦을 확인

예상 결과
- 격자 올리기 22칸, 구간 묶기 16칸; 1분 다운샘플링은 1칸

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

# 한글깨짐 해결
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv('../data/22_열처리.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp').sort_index()

In [ ]:
# [격자 배정(asfreq)과 집계(resample)의 차이점 결합 비교]
# 1. 두 기법으로 각각 변환한 데이터프레임의 컬럼명을 유니크하게 변경한 뒤 `.join()`을 통해 열 방향 결합합니다.
# 2. 동일한 시간 격자에 대조된 결과를 확인하여, 한 구간 안에 들어온 불규칙 다중 신호가 resample 시에는
#    어떻게 병합 요약되고 NaN 개수(22개 vs 16개)를 줄였는지 분석합니다.
# * 데이터 손실 강도와 주기의 정교함을 저울질하여, 정확한 10초 스냅샷이 필요한 제어 로직 분석에는 asfreq를,
#   전체 에너지 흐름 관점의 일관성이 중요할 때는 resample 집계를 선택하는 의사결정을 내릴 수 있습니다.
# * `join()`은 인덱스가 동일한 날짜 기준일 때 가로로 결합해 주는 매우 유용한 판다스 함수입니다.

# 같은 데이터에 격자 올리기와 구간 묶기를 각각 적용
a = df[['제어출력']].asfreq('10s').rename(columns={'제어출력': 'asfreq'})
r = df[['제어출력']].resample('10s').mean().rename(columns={'제어출력': 'resample'})
print(a.join(r).head(8).round(3))
#  asfreq  resample
# timestamp                            
# 2024-03-01 09:00:00   0.426     0.426
# 2024-03-01 09:00:10   0.463     0.463
# 2024-03-01 09:00:20   0.438     0.438
# 2024-03-01 09:00:30   0.445     0.445
# 2024-03-01 09:00:40   0.421     0.421
# 2024-03-01 09:00:50   0.428     0.428
# 2024-03-01 09:01:00   0.405     0.405
# 2024-03-01 09:01:10   0.443     0.443

# 두 결과를 나란히 두고 빈 칸 수를 비교
print('asfreq:', int(a['asfreq'].isna().sum()), '/ resample:', int(r['resample'].isna().sum()))
# asfreq: 22 / resample: 16

# 1분 다운샘플링으로 결측이 크게 줄어듦을 확인
down = df[['제어출력']].resample('1min').mean()
print('1분 다운샘플링:', int(down['제어출력'].isna().sum()))
# 1분 다운샘플링: 1

                     asfreq  resample
timestamp                            
2024-03-01 09:00:00   0.426     0.426
2024-03-01 09:00:10   0.463     0.463
2024-03-01 09:00:20   0.438     0.438
2024-03-01 09:00:30   0.445     0.445
2024-03-01 09:00:40   0.421     0.421
2024-03-01 09:00:50   0.428     0.428
2024-03-01 09:01:00   0.405     0.405
2024-03-01 09:01:10   0.443     0.443
asfreq: 22 / resample: 16
1분 다운샘플링: 1
